In [1]:
import numpy as np 
import pandas as pd 
import sys


In [2]:
import numpy as np
from pathlib import Path
from PIL import Image

import numpy as np
from pathlib import Path
from PIL import Image

def load_dataset(directory, image_size=(512, 512),
                 max_samples=None):

    X = []
    y = []
    rng = np.random.default_rng(42)

    labels = {
        "NORMAL": -1,
        "PNEUMONIA": 1
    }

    for class_name, label in labels.items():

        folder = Path(directory) / class_name

        files = sorted([
            file for file in folder.glob("*")
            if file.suffix.lower() in (".jpeg", ".jpg", ".png")
        ])
        if max_samples is not None:
            size = min(max_samples // 2, len(files))
            files = rng.choice(files, size=size, replace=False)

        for file in files:
            with Image.open(file) as img:
                img = img.convert("L")
                img = img.resize(image_size)

                img = np.asarray(img, dtype=np.float64) / 255.0

                X.append(img.flatten())
                y.append(label)

    return np.array(X), np.array(y)

In [4]:
dataset_path = Path("/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray")

X_train, y_train = load_dataset(dataset_path / "train",max_samples=1000)
X_test, y_test = load_dataset(dataset_path / "test",max_samples=1000)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(1000, 262144)
(1000,)
(624, 262144)
(624,)


In [8]:
from sklearn.svm import LinearSVC

model = LinearSVC(
    C=1.0,
    penalty="l2",
    loss="hinge",
    dual=True,
    max_iter=10000,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [9]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

def evaluate(model, X_test, y_test):

    y_pred = model.predict(X_test)

    print("Precision:", precision_score(y_test, y_pred,
                                       zero_division=0))
    print("Recall:", recall_score(y_test, y_pred,
                                 zero_division=0))
    print("F1-score:", f1_score(y_test, y_pred,
                              zero_division=0))
evaluate(model,X_test, y_test)

Precision: 0.7495029821073559
Recall: 0.9666666666666667
F1-score: 0.8443449048152296
